# Stage 1 - Step 1: Getting the data ready

Before we do anything with encoders or classifiers, we need to make sure the datasets are
actually downloaded correctly and match what the assignment asks for:

- **DTD**: all 47 classes, official partition 1, train/val/test kept separate (not merged)
- **FGVC-Aircraft**: variant-level labels (the hard/fine-grained level, not family or manufacturer), train/val/test kept separate

This notebook just downloads both datasets and prints out enough info that we can eyeball
whether the numbers look right, before we build anything on top of it. If this cell throws
an error or the numbers look off, better to catch it now than after we've cached features
on top of broken data.

Run this in **Google Colab** with a GPU runtime (Runtime > Change runtime type > GPU),
even though this particular notebook doesn't need GPU yet - later ones will.

## 1. Mount Google Drive

We're mounting Drive so the downloaded data (and later, the cached features) survive
between Colab sessions. Otherwise every time the runtime disconnects we'd have to
re-download everything, which with two of us working on this would get old fast.

If you're sharing one Drive folder between the two of you, make sure you're both
pointing at the same `PROJECT_ROOT` path below. If not, that's fine too - just download
independently, it's only a few GB and takes a few minutes.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

# change this if you want it somewhere else in your Drive
PROJECT_ROOT = '/content/drive/MyDrive/cvlab_stage1'
DATA_ROOT = os.path.join(PROJECT_ROOT, 'data')
CACHE_ROOT = os.path.join(PROJECT_ROOT, 'features')   # we'll use this in notebook 2
RESULTS_ROOT = os.path.join(PROJECT_ROOT, 'results')  # we'll use this in notebooks 3-5

for p in [PROJECT_ROOT, DATA_ROOT, CACHE_ROOT, RESULTS_ROOT]:
    os.makedirs(p, exist_ok=True)

print('Project root:', PROJECT_ROOT)
print('Everything below should exist now:')
for p in [DATA_ROOT, CACHE_ROOT, RESULTS_ROOT]:
    print(' -', p, '(exists:', os.path.exists(p), ')')

## 2. Check package versions

Colab comes with torch/torchvision preinstalled, so we don't need to `pip install`
anything for this step. Just checking versions so if something behaves weird later,
we know what we were running.

In [ ]:
import torch, torchvision
print('torch:', torch.__version__)
print('torchvision:', torchvision.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 3. DTD (Describable Textures Dataset)

47 classes, official partition 1. `torchvision`'s `DTD` class already defaults to
`partition=1`, but we're passing it explicitly so it's obvious we're not relying on a
default that could change in a future torchvision version.

This downloads on the first run (~600MB) and just reads from disk on later runs.

In [ ]:
from torchvision.datasets import DTD

dtd_root = os.path.join(DATA_ROOT, 'dtd')

dtd_train = DTD(root=dtd_root, split='train', partition=1, download=True)
dtd_val   = DTD(root=dtd_root, split='val',   partition=1, download=True)
dtd_test  = DTD(root=dtd_root, split='test',  partition=1, download=True)

print('DTD classes:', len(dtd_train.classes))
print('train / val / test sizes:', len(dtd_train), len(dtd_val), len(dtd_test))
print('first few class names:', dtd_train.classes[:5])

# sanity checks - if any of these fail, something's wrong before we go further
assert len(dtd_train.classes) == 47, 'expected 47 DTD classes'
assert dtd_train.classes == dtd_val.classes == dtd_test.classes, 'class lists should match across splits'
print('DTD looks good.')

**What to expect:** DTD partition 1 has 1,880 images in each of train/val/test
(40 images per class x 47 classes). If your numbers are way off from that, stop here
and let's figure out why before continuing.

## 4. FGVC-Aircraft (variant-level)

100 classes. We're using `annotation_level='variant'` - this is the hard setting the
assignment asks for (variant = specific aircraft model like "707-320", not just
"Boeing" or "civil airliner"). This one's a bigger download (~2.7GB), might take a
few minutes.

In [ ]:
from torchvision.datasets import FGVCAircraft

aircraft_root = os.path.join(DATA_ROOT, 'fgvc_aircraft')

aircraft_train = FGVCAircraft(root=aircraft_root, split='train', annotation_level='variant', download=True)
aircraft_val   = FGVCAircraft(root=aircraft_root, split='val',   annotation_level='variant', download=True)
aircraft_test  = FGVCAircraft(root=aircraft_root, split='test',  annotation_level='variant', download=True)

print('Aircraft classes:', len(aircraft_train.classes))
print('train / val / test sizes:', len(aircraft_train), len(aircraft_val), len(aircraft_test))
print('first few class names:', aircraft_train.classes[:5])

assert len(aircraft_train.classes) == 100, 'expected 100 aircraft variant classes'
assert aircraft_train.classes == aircraft_val.classes == aircraft_test.classes
print('Aircraft looks good.')

**What to expect:** roughly 3,334 / 3,333 / 3,333 images in train/val/test
(the official split isn't perfectly balanced per class, that's normal for this dataset).
Class names should look like specific model numbers (e.g. "707-320", "737-200"), not
manufacturer names - that's how we know we're actually on the variant level and not
accidentally on family/manufacturer.

One quirk worth knowing about this dataset for later: the raw images have a thin
banner at the very bottom with a copyright string burned into the pixels. It's a
few pixels tall and doesn't usually cause problems for pretrained encoders, but it's
worth knowing it's there in case anyone asks about preprocessing choices.

## 5. Quick visual sanity check

Not required by the assignment, but let's actually look at a couple of images from
each dataset - cheap way to catch something being loaded wrong (e.g. wrong split,
corrupted download) before we spend time on feature extraction.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for i, ax in enumerate(axes[:2]):
    img, label = dtd_train[i * 200]
    ax.imshow(img)
    ax.set_title(f'DTD: {dtd_train.classes[label]}')
    ax.axis('off')
for i, ax in enumerate(axes[2:]):
    img, label = aircraft_train[i * 200]
    ax.imshow(img)
    ax.set_title(f'Aircraft: {aircraft_train.classes[label]}', fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.show()

## Before moving to the next notebook, check:

- [ ] DTD: 47 classes, ~1880 images in each of train/val/test
- [ ] Aircraft: 100 classes, ~3334/3333/3333 in train/val/test, class names look like model variants
- [ ] The 4 sample images above actually look like textures / aircraft and not something broken
- [ ] `PROJECT_ROOT` on your Drive now has a `data/` folder with both datasets in it

If all of that checks out, move on to **notebook 2 (feature extraction)** - that's where
we load the frozen encoders (ResNet-18 on both datasets, DINOv2 on Aircraft) and cache
the features so we never have to touch the encoders again after that.